# Contexto do case
Imagine que você é um cientista de dados em uma empresa. Seu time está desenvolvendo um sistema de busca de empresas Brasileiras pelo nome fantasia ou razão social. O sistema deverá ser capaz de receber um texto digitado pelo usuário e retornar as empresas mais próximas do texto digitado.

## Saída do modelo:
Precision 1: um acerto é quando a empresa correta está na primeira posição dos resultados retornados.

Precision 5: um acerto é quando a empresa correta está entre as 5 primeiras retornadas.

## 1. Importações e leitura dos dados

In [2]:
import pandas as pd
import numpy as np

import re
import string

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from tqdm import tqdm # para availiação


# 2. Carregando o dataset e Pré-processamento de texto

In [3]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer

# entrada: train.parquet
df = pd.read_parquet("train.parquet")

# primeiras linhas do dataset
df[['user_input', 'razaosocial', 'nome_fantasia', 'cnpj', 'uf']].head()

# baixa as stopwords
nltk.download('stopwords')
nltk.download('rslp')

stop_words = set(stopwords.words('portuguese'))
stemmer = RSLPStemmer()

# inicio do pré processamento e limpeza
def preprocess(text):
    if pd.isnull(text):
        return ""

    # remove pontuações, números, e transforma para minúsculas
    text = re.sub(r'\d+', '', text.lower())
    text = text.translate(str.maketrans('', '', string.punctuation))

    # tokeniza, remove stopwords e aplica stemming
    tokens = text.split()
    tokens = [stemmer.stem(word) for word in tokens if word not in stop_words]

    return " ".join(tokens)

# aplica o pré-processamento nos campos alvo
df['text_target'] = (df['razaosocial'].fillna('') + ' ' + df['nome_fantasia'].fillna('')).apply(preprocess)
df['user_input_clean'] = df['user_input'].apply(preprocess)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package rslp to /root/nltk_data...
[nltk_data]   Unzipping stemmers/rslp.zip.


# 4. Vetorização com TF-IDF e cálculo de similaridade

Neste passo é necessário representar cada texto como um vetor numérico com o TfidfVectorizer e então calcular a similaridade do user_input_clean com todos os text_target usando Cosine Similarity (similaridade por cosseno)

In [4]:
# primeiro vetoriza todos os textos do campo text_target
# (nome fantasia + razão social)
tfidf = TfidfVectorizer()

# ajusta e transforma o campo de busca
tfidf_matrix = tfidf.fit_transform(df['text_target'])

# cria uma função que dado um user_input retorna os top N mais similares
def get_top_matches(user_input_clean, top_n=5):
    # vetoriza o user_input
    query_vec = tfidf.transform([user_input_clean])

    # calcula a similaridade com todos os registros do dataset
    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()

    # pega os indices dos top N mais similares
    top_indices = similarities.argsort()[::-1][:top_n]

    return top_indices, similarities[top_indices]


# Avaliação: precision (top 1 e top 5) do modelo/técnica utilizado
Nesta etapa irá medir se a empresa correta (mesmo CNPJ) está entre os top 1 ou top 5 retornados.

In [ ]:
# Avaliação de top k com dataset completo
top1_hits = 0
top5_hits = 0

for idx, row in tqdm(df.iterrows(), total=len(df)):
    user_input_clean = row['user_input_clean']
    true_cnpj = row['cnpj']

    # pega os índices dos top 5 mais similares
    top_indices, _ = get_top_matches(user_input_clean, top_n=5)

    # verifica se o CNPJ verdadeiro está entre os top
    top_cnpjs = df.iloc[top_indices]['cnpj'].tolist()

    if true_cnpj == top_cnpjs[0]:
        top1_hits += 1
    if true_cnpj in top_cnpjs:
        top5_hits += 1

# calcula as métricas
precision_at_1 = top1_hits / len(df)
precision_at_5 = top5_hits / len(df)

print(f"Precision@1: {precision_at_1:.4f}")
print(f"Precision@5: {precision_at_5:.4f}")


# Avaliação com 100 registros aleatórios

In [6]:
# subset aleatório com 100 registros
df_sample = df.sample(n=100, random_state=42).reset_index(drop=True)

# Vetoriza apenas os dados da amostra
tfidf_sample = TfidfVectorizer()
tfidf_matrix_sample = tfidf_sample.fit_transform(df_sample['text_target'])

def get_top_matches_sample(user_input_clean, top_n=5):
    query_vec = tfidf_sample.transform([user_input_clean])
    similarities = cosine_similarity(query_vec, tfidf_matrix_sample).flatten()
    top_indices = similarities.argsort()[::-1][:top_n]
    return top_indices, similarities[top_indices]

# Avaliação
top1_hits = 0
top5_hits = 0

for idx, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
    user_input_clean = row['user_input_clean']
    true_cnpj = row['cnpj']

    top_indices, _ = get_top_matches_sample(user_input_clean, top_n=5)
    top_cnpjs = df_sample.iloc[top_indices]['cnpj'].tolist()

    if true_cnpj == top_cnpjs[0]:
        top1_hits += 1
    if true_cnpj in top_cnpjs:
        top5_hits += 1

# métricas na amostra
precision_at_1 = top1_hits / len(df_sample)
precision_at_5 = top5_hits / len(df_sample)

print(f"[AMOSTRA] Precision 1: {precision_at_1:.4f}")
print(f"[AMOSTRA] Precision 5: {precision_at_5:.4f}")


100%|██████████| 100/100 [00:00<00:00, 476.66it/s]

[AMOSTRA] Precision@1: 0.6100
[AMOSTRA] Precision@5: 0.8000


# Adicionando filtro por UF

In [7]:
def get_top_matches_by_uf(user_input_clean, uf, df_base, top_n=5):
    # Filtra empresas da mesma UF
    df_filtered = df_base[df_base['uf'] == uf].reset_index(drop=True)

    # se não encontrar empresas na mesma UF, retorna vazio
    if df_filtered.empty:
        return [], [], df_filtered

    # revetoriza os alvos filtrados
    tfidf = TfidfVectorizer()
    tfidf_matrix = tfidf.fit_transform(df_filtered['text_target'])

    # vetoriza a entrada
    query_vec = tfidf.transform([user_input_clean])
    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()

    # pega os top_n mais similares
    top_indices = similarities.argsort()[::-1][:top_n]

    return top_indices, similarities[top_indices], df_filtered


## Avaliação com filtro por UF (subset de 100)

In [9]:
# subset menor para teste
df_sample = df.sample(n=100, random_state=42).reset_index(drop=True)

top1_hits = 0
top5_hits = 0
avaliados = 0

for idx, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
    user_input_clean = row['user_input_clean']
    uf = row['uf']
    true_cnpj = row['cnpj']

    top_indices, _, df_filtered = get_top_matches_by_uf(user_input_clean, uf, df_sample, top_n=5)

    if len(top_indices) == 0:
        continue  # pula se nenhuma empresa da mesma UF

    top_cnpjs = df_filtered.iloc[top_indices]['cnpj'].tolist()

    if true_cnpj == top_cnpjs[0]:
        top1_hits += 1
    if true_cnpj in top_cnpjs:
        top5_hits += 1
    avaliados += 1

# métricas
precision_at_1 = top1_hits / avaliados if avaliados else 0
precision_at_5 = top5_hits / avaliados if avaliados else 0

print(f"[COM UF] Precision 1: {precision_at_1:.4f}")
print(f"[COM UF] Precision 5: {precision_at_5:.4f}")
print(f"Número de exemplos avaliados: {avaliados}")


100%|██████████| 100/100 [00:00<00:00, 184.64it/s]

[COM UF] Precision 1: 0.7900
[COM UF] Precision 5: 0.9100
Número de exemplos avaliados: 100


# Simulação de busca com entrada do usuário (TF-IDF + UF)

In [10]:
def buscar_empresas(user_input_bruto, uf, df_base, top_n=5):
    # pre processamento do texto digitado
    user_input_clean = preprocess(user_input_bruto)

    # filtragem de empresas da mesma UF
    df_filtered = df_base[df_base['uf'] == uf].reset_index(drop=True)

    if df_filtered.empty:
        print(f"Nenhuma empresa encontrada na UF {uf}.")
        return

    # revetoriza com base na UF
    tfidf = TfidfVectorizer()
    tfidf_matrix = tfidf.fit_transform(df_filtered['text_target'])

    # vetoriza o user_input
    query_vec = tfidf.transform([user_input_clean])

    # calcula similaridades
    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_indices = similarities.argsort()[::-1][:top_n]

    # resultados
    print(f"\n🔎 Resultado para busca: \"{user_input_bruto}\" na UF {uf}")
    for rank, i in enumerate(top_indices, 1):
        row = df_filtered.iloc[i]
        print(f"\n🔹 Rank {rank}")
        print(f"Razão Social: {row['razaosocial']}")
        print(f"Nome Fantasia: {row['nome_fantasia']}")
        print(f"CNPJ: {row['cnpj']}")
        print(f"Similaridade: {similarities[i]:.4f}")


## Exemplo de uso:

In [11]:
buscar_empresas("magazine luiza", "SP", df)


🔎 Resultado para busca: "magazine luiza" na UF SP

🔹 Rank 1
Razão Social: MAGAZINE LUIZA S/A
Nome Fantasia: MAGAZINE LUIZA
CNPJ: 47960950045303
Similaridade: 0.9903

🔹 Rank 2
Razão Social: MAGAZINE LUIZA S/A
Nome Fantasia: MAGAZINE LUIZA
CNPJ: 47960950025388
Similaridade: 0.9903

🔹 Rank 3
Razão Social: MAGAZINE LUIZA S/A
Nome Fantasia: MAGAZINE LUIZA
CNPJ: 47960950004623
Similaridade: 0.9903

🔹 Rank 4
Razão Social: MAGAZINE LUIZA S/A
Nome Fantasia: MAGAZINE LUIZA
CNPJ: 47960950045222
Similaridade: 0.9903

🔹 Rank 5
Razão Social: MAGAZINE LUIZA S/A
Nome Fantasia: MAGAZINE LUIZA
CNPJ: 47960950045303
Similaridade: 0.9903
